In [4]:
# -*- coding: utf-8 -*-
"""
Created on Fri Jul 11 10:23:17 2025

@author: silvap1
"""

"""
Cobolt laser routine
--------------------
• Set laser power (mW)
• Blink ON / OFF for user-defined times (s) and cycles

Tested with Cobolt 06-01 series; adjust commands if your model differs.
Requires:  pip install pyserial
"""

import time
import serial
from contextlib import contextmanager


@contextmanager
def cobolt_laser(port: str = "COM5", baud: int = 115200, timeout: float = 0.2):
    """Context-manager that opens the serial port and closes it cleanly."""
    ser = serial.Serial(port, baudrate=baud, timeout=timeout)
    try:
        ser.write(b"\r")        # wake-up
        time.sleep(0.05)
        ser.write(b"cf\r")      # clear faults, just in case
        yield ser
    finally:
        # final safety: ensure emission is OFF and port is closed
        try:
            ser.write(b"!d\r")
        except Exception:
            pass
        ser.close()


def expect_ok(ser):
    """Read one line and require the laser to answer OK."""
    reply = ser.readline().decode(errors="ignore").strip()
    if reply != "OK":
        raise RuntimeError(f"Laser replied '{reply}', not 'OK'")


def set_power(ser, power_mw: int):
    """Set output power in mW."""
    ser.write(f"!p {power_mw}\r".encode())
    expect_ok(ser)


def laser_on(ser):
    ser.write(b"!e\r")
    expect_ok(ser)


def laser_off(ser):
    ser.write(b"!d\r")
    expect_ok(ser)


def blink_sequence(
    port="COM5",
    power_mw=20,
    on_s=1.0,
    off_s=1.0,
    cycles=10,
):
    with cobolt_laser(port) as ser:
        # ----- Enable command-modulation run-mode (only needed once) -----
        # ser.write(b"LASer:RUNMode PowerModulation\r")
        # ser.write(b"LASer:PowerModulation:DIGital:MODulator COMmand\r")
        # expect_ok(ser)
        # -----------------------------------------------------------------
        set_power(ser, power_mw)

        for _ in range(cycles):
            laser_on(ser)
            time.sleep(on_s)
            laser_off(ser)
            time.sleep(off_s)

In [5]:
!pip install pyserial

Defaulting to user installation because normal site-packages is not writeable


In [5]:
if __name__ == "__main__":
    # quick demo: 0.5 s ON / 0.5 s OFF, 100 times at 50 mW
    blink_sequence(
        port="COM8",
        power_mw=20,
        on_s=0.5,
        off_s=0.5,
        cycles=1,
    )

RuntimeError: Laser replied 'Syntax error: illegal command', not 'OK'